# TP 3 — Transfer learning : solution commentée

Solution complète du [TP 3](./enonce.ipynb).

In [4]:
! pwd

/private/tmp/computer-vision-exercices


In [2]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "./data"
IM_MEAN, IM_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

## Exercice 1 — Données

In [3]:
tf = transforms.Compose(
    [
        transforms.Resize(96),
        transforms.ToTensor(),
        transforms.Normalize(IM_MEAN, IM_STD),
    ]
)
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=tf)
test_full = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=tf)
train_ds = Subset(train_full, range(2000))
test_ds = Subset(test_full, range(500))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
print("OK")

  0%|▏                                                                                                                                                                                                   | 197k/170M [00:07<1:53:11, 25.1kB/s]


KeyboardInterrupt: 

## Exercice 2 — MobileNetV2 figé + nouvelle tête

In [3]:
def make_transfer_model(pretrained=True):
    weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    m = models.mobilenet_v2(weights=weights)
    if pretrained:
        for p in m.parameters():
            p.requires_grad = False
    m.classifier[1] = nn.Linear(m.last_channel, 10)
    return m.to(DEVICE)


model = make_transfer_model(pretrained=True)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"paramètres totaux       : {n_total:,}")
print(f"paramètres entraînables : {n_train:,} ({100 * n_train / n_total:.2f} %)")

paramètres totaux       : 2,236,682
paramètres entraînables : 12,810 (0.57 %)


## Exercice 3 — Entraînement de la tête (transfer)

In [4]:
def train_eval(model, train_loader, test_loader, epochs=2):
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    opt = torch.optim.Adam(trainable, lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for e in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
        model.eval()
        correct, n = 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(1) == y).sum().item()
                n += x.size(0)
        acc = correct / n
        history.append(acc)
        print(f"epoch {e + 1}  test acc = {acc:.3f}")
    return history


torch.manual_seed(0)
model_tf = make_transfer_model(pretrained=True)
hist_tf = train_eval(model_tf, train_loader, test_loader, epochs=2)

epoch 1  test acc = 0.640


epoch 2  test acc = 0.664


## Exercice 4 — Comparaison à from-scratch

In [5]:
# From-scratch sur le même budget exact que transfer learning prendrait des heures
# sur CPU (3.4 M paramètres à apprendre). On le démontre sur un sous-ensemble
# plus petit (1000 train / 300 test) et 1 époque — l'écart est déjà criant.
tiny_train = DataLoader(Subset(train_full, range(1000)), batch_size=64, shuffle=True)
tiny_test = DataLoader(Subset(test_full, range(300)), batch_size=128, shuffle=False)

torch.manual_seed(0)
model_scratch = make_transfer_model(pretrained=False)
for p in model_scratch.parameters():
    p.requires_grad = True
hist_scratch = train_eval(model_scratch, tiny_train, tiny_test, epochs=1)

# Pour une comparaison juste, on ré-évalue aussi transfer sur ce même tiny test
model_tf.eval()
correct, n = 0, 0
with torch.no_grad():
    for x, y in tiny_test:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model_tf(x).argmax(1) == y).sum().item()
        n += x.size(0)
tf_on_tiny = correct / n

print()
print(f"transfer learning (sur tiny test) : {tf_on_tiny:.3f}")
print(f"from scratch     (1 ép., 1000 tr.) : {hist_scratch[-1]:.3f}")

epoch 1  test acc = 0.077



transfer learning (sur tiny test) : 0.663
from scratch     (1 ép., 1000 tr.) : 0.077


**Discussion** : le modèle pré-entraîné part avec des filtres convolutifs qui ont déjà appris à détecter bords, textures et motifs sur ImageNet (1.2 M d'images). On hérite donc de toute cette représentation visuelle et on n'apprend plus que la projection vers nos 10 classes. From scratch sur 1 000 images et 1 époque, le modèle n'a quasiment rien appris — il est proche du hasard (10 %). C'est exactement pour cela que **le transfer learning est la norme dès qu'on a peu de données**.

Pour aller plus loin (en dehors du temps du TP) :

- **Fine-tuning partiel** : dégeler les 2-3 derniers blocs du backbone et continuer avec un `lr` 10× plus petit.
- Tester d'autres backbones (`resnet18`, `efficientnet_b0`).
- Augmentation de données + scheduler de learning rate.